In [1]:
from datasets import load_dataset

# Data are loaded to fine-tuning from already prepared synthetic audio
dataset = load_dataset("audiofolder", data_dir=f"../data/training")
dataset

/home/c100rczyk/anaconda3/envs/stt_evaluation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset.column_names['train']

['audio',
 'text',
 'conversation',
 'turn',
 'speaker',
 'voice',
 'duration',
 'subdomain',
 'terms']

In [4]:
# Feature extractor for pre-processing raw audio inputs into spectogram
from transformers import WhisperFeatureExtractor
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

In [5]:
feature_extractor

WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

In [6]:
# Whisper Tokenizer (For transcription part)
from transformers import WhisperTokenizer

# This tokenizer is for decoding transcription of audio into token ID's.
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="polish", task="transcribe")

In [7]:
input_str = dataset['train'][0]['text']
input_str

'Dzień dobry, proszę wejść. Kto do nas dzisiaj przyszedł?'

In [8]:
labels = tokenizer(input_str).input_ids
print(labels)

[50258, 50269, 50359, 50363, 35, 3283, 5248, 35884, 11, 39677, 321, 44536, 13, 591, 1353, 360, 5382, 25772, 6541, 749, 11312, 1221, 30, 50257]


In [9]:
decode_special = tokenizer.decode(labels, skip_special_tokens=False)
decode_special

'<|startoftranscript|><|pl|><|transcribe|><|notimestamps|>Dzień dobry, proszę wejść. Kto do nas dzisiaj przyszedł?<|endoftext|>'

In [10]:
dataset['train'][0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7c225488f010>,
 'text': 'Dzień dobry, proszę wejść. Kto do nas dzisiaj przyszedł?',
 'conversation': 'train_004',
 'turn': 1,
 'speaker': 'doctor',
 'voice': 'cv_k_2c818dfafa',
 'duration': 2.89,
 'subdomain': 'pediatra',
 'terms': []}

In [11]:
#DATA:

feature_extractor(dataset['train'][0]['audio']["array"], sampling_rate=dataset['train'][0]['audio']["sampling_rate"]).input_features[0]

array([[-0.4267155 , -0.16524196,  0.10797912, ..., -0.7088783 ,
        -0.7088783 , -0.7088783 ],
       [-0.32936287, -0.22011924,  0.28342706, ..., -0.7088783 ,
        -0.7088783 , -0.7088783 ],
       [-0.20299542,  0.03974974,  0.30156666, ..., -0.7088783 ,
        -0.7088783 , -0.7088783 ],
       ...,
       [-0.7088783 , -0.7088783 , -0.7088783 , ..., -0.7088783 ,
        -0.7088783 , -0.7088783 ],
       [-0.7088783 , -0.7088783 , -0.7088783 , ..., -0.7088783 ,
        -0.7088783 , -0.7088783 ],
       [-0.7088783 , -0.7088783 , -0.7088783 , ..., -0.7088783 ,
        -0.7088783 , -0.7088783 ]], shape=(80, 3000), dtype=float32)

In [12]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch['input_features'] = feature_extractor(audio["array"], sampling_rate=audio['sampling_rate']).input_features[0]

    batch["labels"] = tokenizer(batch["text"]).input_ids

    return batch


In [13]:
# Prepare features and tokenization for data that we want to train on.
dataset = dataset.map(prepare_dataset)

In [14]:
# Load model for Fine-tuning with proper checkpoint.
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1441.55it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [15]:
# Because it is known 
model.generation_config.language = "polish"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


In [16]:
# Function that from a list of single samples creates one batch of tensors.
import numpy as np
import torch

# Fill value for labels. PyTorch cross-entropy skips positions with -100,
# so the model is not scored on the fill, only on real tokens.
IGNORE_INDEX = -100

# ID of <|startoftranscript|>, the token every decoder input begins with.
DECODER_START_ID = model.config.decoder_start_token_id


def prepare_batch(samples):
    """Join single samples into one batch of tensors for the model.

    samples: list of dataset rows after `prepare_dataset`. Only two fields are read:
        "input_features" - log-mel spectrogram, 80 x 3000
        "labels"         - token IDs of the transcription

    Returns a dict that can be passed straight to the model:
        "input_features" - float32 tensor, shape (batch, 80, 3000)
        "labels"         - int64 tensor, shape (batch, longest transcription in the batch)
    """
    # 1. Spectrograms. The feature extractor already filled every recording up to 30 s,
    #    so all spectrograms have the same shape and can be stacked as they are.
    #    (Going through numpy is much faster than building a tensor from nested lists.)
    input_features = torch.stack(
        [torch.from_numpy(np.asarray(sample["input_features"], dtype=np.float32)) for sample in samples]
    )

    # 2. Transcriptions.
    token_ids = [torch.as_tensor(sample["labels"], dtype=torch.long) for sample in samples]

    # The model builds the decoder input on its own: it shifts the labels one position
    # to the right and puts <|startoftranscript|> in front. The tokenizer has already
    # put this token in front too, so it is cut off here. Without that the decoder
    # would get it twice.
    token_ids = [ids[1:] if ids[0] == DECODER_START_ID else ids for ids in token_ids]

    # Transcriptions differ in length and a tensor has to be a rectangle. Start from
    # a tensor filled with IGNORE_INDEX, as wide as the longest transcription,
    # and write each transcription into the beginning of its row.
    longest = max(len(ids) for ids in token_ids)
    labels = torch.full((len(samples), longest), IGNORE_INDEX, dtype=torch.long)
    for row, ids in enumerate(token_ids):
        labels[row, : len(ids)] = ids

    return {"input_features": input_features, "labels": labels}

In [17]:
# Check on the first 4 samples: shapes of both tensors and how the fill looks.
batch = prepare_batch([dataset["train"][i] for i in range(4)])

print(batch["input_features"].shape, batch["input_features"].dtype)
print(batch["labels"].shape, batch["labels"].dtype)
batch["labels"]

torch.Size([4, 80, 3000]) torch.float32
torch.Size([4, 58]) torch.int64


tensor([[50269, 50359, 50363,    35,  3283,  5248, 35884,    11, 39677,   321,
         44536,    13,   591,  1353,   360,  5382, 25772,  6541,   749, 11312,
          1221,    30, 50257,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
        [50269, 50359, 50363,    35,  3283,  5248, 35884,    11, 43916,   360,
         28359,    13,  1407,  3492,   591, 12584,    11,   463,  1017, 46722,
            13, 12210,   261,  3689,   284,  1805,   463, 27062,   453,  1611,
         24012,  1611,  3689, 15724,    11,   257,   261,   572,  1344,  1111,
           532,  3992,  1221,  3244,   710, 28695, 14875,   443,   741, 34573,
          1696,  1221,  3244,  7949,   344,  5738,    13, 50257],
        [50269, 50359, 50363,    34,  1381,  7753,    11,   591, 12584,    13,

In [18]:
# Metrics for evaluation during training: WER and CER.
import sys

# metrics.py lives in ../evaluation. The folder goes to the end of the search path, so that
# no installed package is replaced by a file from it (the folder has its own evaluate.py).
if "../evaluation" not in sys.path:
    sys.path.append("../evaluation")
import metrics


def compute_metrics(prediction):
    """Compare transcriptions written by the model with the reference ones.

    prediction: what the trainer passes after an evaluation run. The trainer has to generate
    text for that (predict_with_generate=True):
        prediction.predictions - token IDs written by the model, shape (recordings, tokens)
        prediction.label_ids   - token IDs of the reference transcriptions, filled with -100

    Returns error rates as fractions (0.25 = 25%), lower is better:
        "wer"         - word error rate: wrong, missing and extra words / words in the references
        "wer_written" - WER after both texts are rewritten to one written form
                        ("pięćset złotych" = "500 zł"), so only wrong words are left
        "cer"         - like "wer", counted on characters
    """
    def to_texts(token_ids):
        # -100 is not a token, so the tokenizer cannot decode it. The pad token goes in its place:
        # it is a special token and is dropped together with <|startoftranscript|>, <|pl|> etc.
        token_ids = np.where(token_ids == IGNORE_INDEX, tokenizer.pad_token_id, token_ids)
        return tokenizer.batch_decode(token_ids, skip_special_tokens=True)

    transcripts = to_texts(prediction.predictions)
    references = to_texts(prediction.label_ids)

    # Lowercasing, removing punctuation and counting errors happen inside metrics.score.
    scores = [metrics.score(reference, transcript) for reference, transcript in zip(references, transcripts)]

    # Errors and lengths are summed over all recordings and divided at the end. An average
    # of per-recording rates would let a wrong "Tak." weigh as much as a wrong long turn.
    def rate(errors, length):
        return sum(score[errors] for score in scores) / max(sum(score[length] for score in scores), 1)

    return {
        "wer": rate("word_errors", "words"),
        "wer_written": rate("word_errors_written", "words_written"),
        "cer": rate("char_errors", "chars"),
    }

In [19]:
# Training arguments: how long and how fast the model learns, when it is evaluated and what is saved.
from transformers import Seq2SeqTrainingArguments

# Evaluation during training needs the validation split (data/training/validation/).
# Without it the model is only trained and the last checkpoints are kept.
HAS_VALIDATION = "validation" in dataset

training_args = Seq2SeqTrainingArguments(
    # Checkpoints go to ft-stt-pipeline/checkpoints/ (kept out of git).
    output_dir="checkpoints/whisper-small",

    # --- How long and how fast the model learns ---
    num_train_epochs=2,                # one epoch = every training clip seen once
    per_device_train_batch_size=2,      # clips per step; lower it when the GPU runs out of memory
    learning_rate=1e-5,                 # low, so the model adapts without forgetting what it already knows
    warmup_steps=0.1,                   # over the first 10% of steps the learning rate grows from 0 to 1e-5,
    lr_scheduler_type="linear",         # then falls in a straight line back to 0
    fp16=torch.cuda.is_available(),     # half precision on a GPU: less memory, faster steps

    # --- Evaluation on the validation split ---
    eval_strategy="epoch" if HAS_VALIDATION else "no",
    per_device_eval_batch_size=8,
    predict_with_generate=True,         # the model writes whole transcriptions; compute_metrics needs text
    generation_max_length=225,          # longest transcription it may write, in tokens

    # --- Checkpoints ---
    save_strategy="epoch",              # same moment as evaluation, so every checkpoint has its WER
    save_total_limit=2,                 # one checkpoint takes about 3 GB; older ones are deleted
    load_best_model_at_end=HAS_VALIDATION,  # after training go back to the checkpoint with the lowest WER
    metric_for_best_model="wer",
    greater_is_better=False,

    # --- Logging ---
    logging_steps=10,                    # training loss is shown every 5 steps
    report_to="none",                   # no experiment tracker yet
)

In [20]:
# Trainer: joins the model, the data and the functions defined above, and runs the training loop.
from transformers import Seq2SeqTrainer, WhisperProcessor

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"] if HAS_VALIDATION else None,
    data_collator=prepare_batch,
    compute_metrics=compute_metrics,
    # Feature extractor and tokenizer are saved into every checkpoint next to the weights, so a
    # checkpoint directory can be loaded on its own (evaluation/evaluate.py --model <checkpoint directory>).
    processing_class=WhisperProcessor(feature_extractor, tokenizer),
)

In [21]:
trainer.train()

Step,Training Loss
10,0.663645
20,0.426980
30,0.386450
40,0.278240
50,0.387919
60,0.207032
70,0.264817
80,0.132987
90,0.056130
100,0.077732


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.62s/it]


TrainOutput(global_step=146, training_loss=0.21512677571544908, metrics={'train_runtime': 44.194, 'train_samples_per_second': 6.607, 'train_steps_per_second': 3.304, 'total_flos': 8.426693689344e+16, 'train_loss': 0.21512677571544908, 'epoch': 2.0})